#### Imports

In [ ]:
import pandas as pd
import numpy as np
import time
import pyodbc
import pickle
import warnings
import os
from tensorflow.keras.models import load_model
import winsound 

warnings.filterwarnings('ignore')   
print("--- INITIALIZING FACTORY INFERENCE ENGINE ---")

#### Define the Target Serial

In [ ]:
SELECTED_MACHINE = input("Enter the Machine Serial Number to analyze (e.g., 0521760): ").strip()
print(f"✅ Target Machine Set To: {SELECTED_MACHINE}")

#### Azure SQL & Path Configuration

In [ ]:
# 1. AZURE SQL CONFIGURATION
AZURE_SERVER = 'kreseakreiotprdsrv.database.windows.net'
AZURE_DATABASE = 'KRESEAKREIOTPRD'
AZURE_USERNAME = 'IOTAdmin'
AZURE_PASSWORD = 'oKuvodump5JNG7dM' 
AZURE_TABLE_NAME = 'MBP_ControllerData' 

# 2. PATHS TO TRAINED FILES
MODEL_PATH = r"C:\Users\DeelakaD\OneDrive - MAS Holdings (Pvt) Ltd\Documents\GitHub\Preventing-Mechanism\sewing_machine_predictive_lstm.h5"
SCALER_PATH = r'C:\Users\DeelakaD\OneDrive - MAS Holdings (Pvt) Ltd\Documents\GitHub\Preventing-Mechanism\sewing_scaler.pkl'
ENCODER_PATH = r'C:\Users\DeelakaD\OneDrive - MAS Holdings (Pvt) Ltd\Documents\GitHub\Preventing-Mechanism\sewing_encoder.pkl'

# 3. DATABASE DRIVER
driver = '{ODBC Driver 17 for SQL Server}'
conn_str = f'DRIVER={driver};SERVER={AZURE_SERVER};PORT=1433;DATABASE={AZURE_DATABASE};UID={AZURE_USERNAME};PWD={AZURE_PASSWORD}'

#### Load Baseline Model and Translators

In [ ]:
try:
    model = load_model(MODEL_PATH)
    with open(SCALER_PATH, 'rb') as f:
        scaler = pickle.load(f)
    with open(ENCODER_PATH, 'rb') as f:
        encoder = pickle.load(f)
    print("✅ AI Model and Translators Loaded Successfully.")
except Exception as e:
    print(f"❌ ERROR LOADING FILES: {e}")

#### Logic Engines

In [ ]:
# Rule-Based Solution Engine
maintenance_solutions = {
    "Needle Breakage": "SOLUTION: Immediately replace the needle and check the needle guard alignment.",
    "Thread Jam": "SOLUTION: Clear the bobbin case, check thread tension, and clean the feed dogs.",
    "Blade Blunt": "SOLUTION: Schedule a replacement of the upper and lower cutting blades within 1 hour.",
    "High Foot Pressure": "SOLUTION: Reduce presser foot pressure dial by 2 turns.",
    "Healthy Baseline": "STATUS: Machine operating normally. No action required."
}

def parse_vibration_to_bands(vib_str):
    if pd.isna(vib_str): return {}
    parts = str(vib_str).split(',')
    res = {}
    try:
        for i in range(0, len(parts)-1, 2):
            f_start = int(parts[i])
            f_end = f_start + 10
            res[f"{f_start}-{f_end}Hz"] = int(parts[i+1])
    except Exception:
        pass
    return res

# Define the exact 69 columns expected by the LSTM
electrical_cols = [ 
    'machineVoltageMean', 'machineVoltageMin', 'machineVoltageMax',
    'machineCurrentMean', 'machineCurrentMin', 'machineCurrentMax',
    'machinePowerMean', 'machinePowerMin', 'machinePowerMax'
]
vibration_bands = [f"{i}-{i+10}Hz" for i in range(10, 610, 10)]
ALL_69_FEATURES = electrical_cols + vibration_bands

#### The Live Monitoring Loop

In [ ]:
last_processed_time = None

try:
    print(f"🔄 Connecting to Azure SQL Database: {AZURE_DATABASE}...")
    with pyodbc.connect(conn_str) as conn:
        print(f"✅ Connection Verified. Monitoring Machine: {SELECTED_MACHINE}\n")
        
        # Query to pull the absolute newest 5 records for the chosen machine
        query = f"""
            SELECT TOP 5 * FROM {AZURE_TABLE_NAME} 
            WHERE machineSerialNumber = '{SELECTED_MACHINE}' 
            ORDER BY dateTime DESC
        """
        
        while True:
            df_live = pd.read_sql(query, conn)
            
            if not df_live.empty and len(df_live) >= 5:
                # 1. EXTRACT LITERAL DATETIME FROM AZURE
                db_raw_timestamp = df_live['dateTime'].iloc[0]
                
                # 2. CONVERT FOR INTERNAL LOGIC AND LOCAL DISPLAY
                last_record_utc = pd.to_datetime(db_raw_timestamp)
                # Adding 5:30 offset for Sri Lanka
                sl_record_time = last_record_utc + pd.Timedelta(hours=5, minutes=30)

                # 3. Capture Current System Time
                current_local_time = pd.Timestamp.now()
                
                # 3. ONLY RUN IF A FRESH RECORD HAS ARRIVED
                if sl_record_time != last_processed_time:
                    last_processed_time = sl_record_time

                    
                    
                    # --- AI PRE-PROCESSING ---
                    # Reverse to chronological order (asc) for the LSTM window
                    df_live_sorted = df_live.iloc[::-1].reset_index(drop=True)
                    vibs = pd.DataFrame(df_live_sorted['machineVibration'].apply(parse_vibration_to_bands).tolist()).fillna(0)
                    extras = df_live_sorted[electrical_cols]
                    df_window = pd.concat([extras, vibs], axis=1).fillna(0)
                    
                    # Align 69 features to solve StandardScaler attribute issues
                    for col in ALL_69_FEATURES:
                        if col not in df_window.columns:
                            df_window[col] = 0
                    df_window = df_window[ALL_69_FEATURES] 
                    
                    # --- INFERENCE ---
                    X_live_scaled = scaler.transform(df_window.values)
                    X_inference = np.array([X_live_scaled])
                    prediction_probs = model.predict(X_inference, verbose=0)
                    predicted_state = encoder.inverse_transform([np.argmax(prediction_probs)])[0]
                    
                    # --- FINAL DASHBOARD OUTPUT ---
                    print(f"\n--- [MACHINE: {SELECTED_MACHINE}] ---")
                    # Prints the exact string from the dateTime column (truncated to seconds)
                    print(f"AZURE DB Last Record TIMESTAMP (UTC): {str(db_raw_timestamp).split('.')[0]}") 
                    # Prints full local date and time for Sri Lanka
                    print(f"LOCAL TIME (SRI LANKA) : {sl_record_time.strftime('%Y-%m-%d %H:%M:%S')}")
                    print(f"CURRENT LOCAL TIME : {current_local_time.strftime('%Y-%m-%d %H:%M:%S')}")
                    print(f"PREDICTED MACHINE STATE : {predicted_state}")
                    
                    if predicted_state in maintenance_solutions:
                        print(f"{maintenance_solutions[predicted_state]}")
                
            elif df_live.empty:
                print(f"❌ No records found for Serial {SELECTED_MACHINE}.", end="\r")
            
            # Polling frequency
            time.sleep(1)

except Exception as e:
    print(f"\n❌ SCRIPT ERROR: {e}")